# From model to decision

Data Scientist track, lesson DS16, the capstone. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** frame a problem with a named stakeholder, a decision and a success definition; model it and evaluate it honestly with baselines, an error analysis and limits; describe deployment, monitoring and a retraining trigger on paper; end with a decision memo and an AI verification log.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
The cells before "Your work" give you the churn table, the three parts, the day-7 model and the offer economics of the track. Work in the empty cells and the text cells under "Your work".
The reference answer starts at the last heading: open it only after you finish. It reads only the given cells, so it gives the same output whether or not you did your work first. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 2026       # the seed of the whole Data Scientist track

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)

# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")

## The three parts and the day-7 model

In [ ]:
# The columns, the three parts and the day-7 model of lessons DS2 to DS5. Run this cell as it is.
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
X, y = table[STATIC + WEEK1], table["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)       # 13,362 and 4,454 people
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train)   # 10,021 and 3,341


def logit(cat=STATIC, num=WEEK1):
    """The model of lesson DS2: one-hot text columns, scaled number columns, then a logistic regression (no number columns: the signup rung)."""
    parts = [("cat", OneHotEncoder(handle_unknown="ignore"), cat)] + ([("num", StandardScaler(), num)] if num else [])
    return make_pipeline(ColumnTransformer(parts), LogisticRegression(max_iter=1000))


day7 = logit().fit(X_fit, y_fit)                                  # fitted on the 10,021 fitting people, as in lesson DS4
p_val, p_test = day7.predict_proba(X_val)[:, 1], day7.predict_proba(X_test)[:, 1]
yv, yt = y_val.to_numpy(), y_test.to_numpy()
print(len(X_fit), len(X_val), len(X_test), round(float(roc_auc_score(yt, p_test)), 3))

## The offer economics and three helpers

In [ ]:
# The offer economics of lesson DS4: assumptions of the example, not facts about the app. Run this cell as it is.
OFFER_COST, RETURN_VALUE, RETURN_RATE = 2.0, 12.0, 0.25      # an offer costs $2; a leaver who gets one comes back with chance 0.25, worth $12


def value_per_1000(flag, y, return_rate=RETURN_RATE):
    """What the offer earns for every 1,000 people against sending nobody: a leaver offered gains return_rate x $12 less the $2, a stayer offered costs the $2."""
    flag, y = np.asarray(flag), np.asarray(y)
    return 1000 * ((flag & (y == 1)).sum() * (return_rate * RETURN_VALUE - OFFER_COST) - (flag & (y == 0)).sum() * OFFER_COST) / len(y)


def value_gain(flag_a, flag_b, y, resamples=2000):
    """Value of rule a minus value of rule b (b = nobody when None) for every 1,000 people, with a 95 percent bootstrap interval; both on the same resamples."""
    rng = np.random.default_rng(SEED)
    y = np.asarray(y)
    a = np.asarray(flag_a)
    b = np.zeros(len(y), dtype=bool) if flag_b is None else np.asarray(flag_b)
    gains = []
    for _ in range(resamples):
        i = rng.integers(0, len(y), len(y))
        gains.append(value_per_1000(a[i], y[i]) - value_per_1000(b[i], y[i]))
    low, high = np.percentile(gains, [2.5, 97.5])
    return [round(float(value_per_1000(a, y) - value_per_1000(b, y)), 1) + 0.0, round(float(low), 1) + 0.0, round(float(high), 1) + 0.0]


def auc_gain(p_new, p_base, y_true, resamples=2000):
    """ROC AUC of one score minus another on the same people, with a 95 percent bootstrap interval; both are scored on the same resamples."""
    rng = np.random.default_rng(SEED)
    y_true, a, b = np.asarray(y_true), np.asarray(p_new), np.asarray(p_base)
    gains = []
    for _ in range(resamples):
        i = rng.integers(0, len(y_true), len(y_true))
        gains.append(roc_auc_score(y_true[i], a[i]) - roc_auc_score(y_true[i], b[i]))
    low, high = np.percentile(gains, [2.5, 97.5])
    return [round(float(roc_auc_score(y_true, a) - roc_auc_score(y_true, b)), 3) + 0.0, round(float(low), 3) + 0.0, round(float(high), 3) + 0.0]

## Your work

Start a timer for 3 hours, 4 at most. Work through the seven deliverables of the lesson in order.

### 1. Your framing page

Double-click here and write it: the stakeholder, the decision, the success definition, the target, the prediction moment and the assumptions.

### 2. Your baselines, model and evaluation

In [ ]:
# your work here

### 3. Your error analysis

In [ ]:
# your work here

### 4. Your deployment, monitoring and retraining plan

Double-click here and write it. Use a cell below to try a check on the past.

In [ ]:
# your work here

### 5. Your decision memo

Double-click here and write it.

### 6. Your AI verification log

Double-click here and write it: the context pack, then one row for every AI output you used.

### 7. Your explain-back

Double-click here and write it.

## Reference answer: open only after you finish

The cells below read only the given cells, so they give the same output whatever your own cells did.

### Baselines, the rule and what it earns

In [ ]:
# Baselines and the day-7 model on the 4,454 held-out people, each gain with a paired interval.
signup = logit(num=[]).fit(X_fit, y_fit)                          # the signup-columns rung of lesson DS2
p_signup = signup.predict_proba(X_test)[:, 1]
p_flat = np.full(len(yt), y_fit.mean())
print([round(float(roc_auc_score(yt, p)), 3) for p in (p_flat, p_signup, p_test)])
print(auc_gain(p_test, p_signup, yt), auc_gain(p_test, p_flat, yt))

# The rule: offer when the chance is at least the threshold that earns most on the validation people. The held-out people are used once.
grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
best = float(max(grid, key=lambda t: value_per_1000(p_val >= t, yv)))
model_rule = p_test >= best
paid_social = (X_test["channel"] == "paid_social").to_numpy()      # a rule that fits on a sticky note
print(best, int(model_rule.sum()), round(float(model_rule.mean()), 3), int(paid_social.sum()))
for name, rule in [("model rule", model_rule), ("paid-social rule", paid_social), ("everybody", np.ones(len(yt), dtype=bool))]:
    print(f"{name}: {value_per_1000(rule, yt):.1f}")
print(value_gain(model_rule, None, yt), value_gain(paid_social, None, yt), value_gain(model_rule, paid_social, yt))

### Error analysis

In [ ]:
# Error analysis: who is offered, and which leavers does the rule miss?
errors = pd.DataFrame({"channel": X_test["channel"].to_numpy(), "platform": X_test["platform"].to_numpy(), "churned": yt, "offered": model_rule})


def slice_table(frame, col):
    """For each level of `col`: people, churn rate, share offered and share of leavers offered (recall)."""
    rows = {}
    for level, s in frame.groupby(col):
        rows[level] = [len(s), s["churned"].mean(), s["offered"].mean(), s.loc[s["churned"] == 1, "offered"].mean()]
    return pd.DataFrame.from_dict(rows, orient="index", columns=["people", "churn rate", "offered", "leavers offered"]).round(3)


print(slice_table(errors, "channel").to_string())
print(slice_table(errors, "platform").to_string())
print(slice_table(errors[errors["channel"] == "paid_social"], "platform").to_string())

### What the offer earns at other return rates and channel mixes

In [ ]:
# What the rule earns depends on the return rate, an assumption: the rate at which it breaks even, and what it earns at other rates.
tp, fp = int((model_rule & (yt == 1)).sum()), int((model_rule & (yt == 0)).sum())
break_even = (OFFER_COST + OFFER_COST * fp / tp) / RETURN_VALUE
print(tp, fp, round(break_even, 3))
print({rate: round(float(value_per_1000(model_rule, yt, rate))) for rate in (0.15, 0.20, 0.25, 0.30)})
# And at another channel mix: the value for every 1,000 paid-social people, their share of the held-out people, and the value at the share after the campaign (0.093).
per_paid_social = value_per_1000(paid_social[paid_social], yt[paid_social])
print(round(float(per_paid_social)), round(float(paid_social.mean()), 3), round(float(per_paid_social * 0.093)))

### A monitoring back-test

In [ ]:
# Monitoring on paper, tried on the past: fit on the signups before March 2, then score each full week of signups from March 2 once its 30 days are over.
signup_day = pd.to_datetime(table["signup_date"])
early = table[signup_day < pd.Timestamp("2026-03-02")]
old_model = logit().fit(early[STATIC + WEEK1], early["churned"])
rows = []
for week, g in table[signup_day >= pd.Timestamp("2026-03-02")].groupby(signup_day.dt.to_period("W-SUN").dt.start_time):
    p = old_model.predict_proba(g[STATIC + WEEK1])[:, 1]
    rows.append([str(week.date()), len(g), g["churned"].mean(), p.mean(), roc_auc_score(g["churned"], p), (g["channel"] == "paid_social").mean()])
weekly = pd.DataFrame(rows, columns=["week", "people", "churned", "mean chance", "AUC", "paid social"]).set_index("week")
weekly["gap"] = weekly["mean chance"] - weekly["churned"]
print(weekly.round(3).to_string())
fires = (weekly["gap"].abs() > 0.05) | (weekly["AUC"] < 0.60)             # the trigger: a gap over 5 points or a weekly AUC under 0.60
print(len(early), len(weekly), round(float(weekly["gap"].abs().max()), 3), round(float(weekly["AUC"].min()), 3), int(fires.sum()))

### The age of the model

In [ ]:
# How stale can the model be? Fit before March and fit before May, both scored on the May signups.
may = table[signup_day >= pd.Timestamp("2026-05-01")]
for label, fit_on in [("fit before March", early), ("fit before May", table[signup_day < pd.Timestamp("2026-05-01")])]:
    p = logit().fit(fit_on[STATIC + WEEK1], fit_on["churned"]).predict_proba(may[STATIC + WEEK1])[:, 1]
    print(label, len(fit_on), round(float(roc_auc_score(may["churned"], p)), 3), round(float(p.mean()), 3))
print(len(may), round(float(may["churned"].mean()), 3))

### An input check

In [ ]:
# An input check that would have caught a silent change: iOS people with a paywall event in their first 7 days, by signup week, counting both event names and the old name only.
rename = con.execute("""WITH people AS (
  SELECT u.user_id, u.signup_date, s.event_ts + INTERVAL 7 DAY AS cutoff
  FROM users AS u JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-04-20' AND DATE '2026-05-31' AND s.platform = 'ios'
)
SELECT CAST(date_trunc('week', p.signup_date) AS DATE) AS week, COUNT(DISTINCT p.user_id) AS people,
       COUNT(DISTINCT p.user_id) FILTER (WHERE e.event_name IN ('view_paywall', 'paywall_viewed')) AS both_names,
       COUNT(DISTINCT p.user_id) FILTER (WHERE e.event_name = 'view_paywall') AS old_name_only
FROM people AS p
LEFT JOIN events AS e ON e.user_id = p.user_id AND e.event_ts <= p.cutoff AND e.received_at <= p.cutoff
GROUP BY 1 ORDER BY 1""").df()
rename["share, both names"] = (rename["both_names"] / rename["people"]).round(3)
rename["share, old name only"] = (rename["old_name_only"] / rename["people"]).round(3)
print(rename.drop(columns=["both_names", "old_name_only"]).to_string(index=False))

### Two checks for the AI log

In [ ]:
# Two checks of the AI log, run by hand: does the name exist, and does scaling before the split change the number?
import sklearn.metrics as sk_metrics

print(hasattr(sk_metrics, "roc_auc_score"), hasattr(sk_metrics, "threshold_cost_score"))
numbers = X[WEEK1]
scaled = pd.concat([X[STATIC], (numbers - numbers.mean()) / numbers.std(ddof=0)], axis=1)           # the assistant's way: scale the whole table, then split
a, b, ya, yb = train_test_split(scaled, y, test_size=0.25, random_state=SEED, stratify=y)
plain = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", "passthrough", WEEK1)]), LogisticRegression(max_iter=1000)).fit(a, ya)
print(round(float(roc_auc_score(yb, plain.predict_proba(b)[:, 1])), 4), round(float(roc_auc_score(y_test, logit().fit(X_train, y_train).predict_proba(X_test)[:, 1])), 4))

## Run all from the top

Restart the session, run every cell from the first, then compare the numbers of the reference answer with the ones in your memo.